# Lab 1

## Exercise 1

1. The agent needs memory of which squares it has cleaned.
2. The agent needs a rule to suck whenever the current square is dirty.
3. The agent needs a rule to move to unexplored squares to finish the job.
4. The agent needs a stopping rule once it knows everything is clean to avoid movement penalty. 

## Exercise 2

**Playing soccer**:
1. Performance Measure: goals scored minus goals conceeded, winning the match
2. Environment: pitch, ball, nets, teammates, opponents, referee, weather
3. Actuators: legs to run and kick, feet and hand to strike the ball, body to shield it, voice to call to teammates
4. Sensors: eyes/cameras to track the ball and players, ears to hear teammate

**Performing a high jump**:
1. Performance Measure: height of the bar from ground cleared successfully and maximized, penalty for knocking the bar off
2. Environment: runway, bar, landing mat, weather
3. Actuators: legs to run and jump, body and arms for movement
4. Sensors: eyes/camera to track the bar height

**Knitting a sweater**:
1. Performance Measure: correct size and shape, even stitches, no dropped stitches
2. Environment: needles, warn
3. Actuators: hands to knit
4. Sensors: eyes to see, hands to feel

## Exercise 3

**Playing soccer**:
1. Simple reflex: If ball is at feet then kick it. 
2. Model-based: Remembers where teammates were a moment ago even when it can't see them now, and tracks the score and clock.
3. Goal-based: Adds a goal 'score', instead of just reacting it plans a passing move towards the net. 
4. Utility-based: Adds a utility function for when there are competing or partial goals. Weights a risky shot against a safe pass, picks whichever one maximizes expected payoff. 

PM: receives 1 if ball is kicked else 0. 

**Performing a high jump**:
1. Simple reflex: acts on the current percept only. At the take-off mark it fires a jump reflex, no memory of previous attempts or awareness of its run-up.
2. Model-based: keeps internal state through the attempt: stride count, approach speed, position relative to the bar, and crucially its own body orientation in the air, which it cannot directly see. It tracks how the jump is unfolding.
3. Goal-based: holds the explicit goal "clear the bar at this height", and reasons about which take-off point, angle and technique will get it over, planning the approach to achieve clearance rather than just reacting.
4. Utility-based: weighs competing concerns: probability of clearing versus energy spent versus injury risk, or in a competition, which height to attempt to maximise final placing. It picks the option with highest expected utility.

PM: receives 1 if jumps above ground else 0. 

**Knitting a sweater**:
1. Simple reflex: reads the current stitch or pattern line and executes it ("this stitch is a knit, so knit"), with no memory of where it is overall.
2. Model-based: tracks internal state it can't see all at once: which row it's on, the running stitch count, tension, and progress against the pattern. This is what stops it losing its place.
3. Goal-based: holds the goal of a finished sweater of the right shape and size, and plans the sequence of rows and shaping (increases and decreases) needed to get there.
4. Utility-based: trades off neatness against speed against yarn used against fit, choosing actions that maximise overall satisfaction with the finished garment.

PM: receives 1 if a knit is made else 0. 

## Exercise 4

In [5]:
# Percept: [location, status], e.g. ["A", "Dirty"]
# Actions: "Left", "Right", "Suck"

def vacuum_agent(percept):
    location, status = percept
    if status == 'Dirty':
        return 'Suck'
    elif location == 'A':
        return 'Right'
    else:
        return 'Left'
        
class VacuumWorld:
    def __init__(self, dirt, location='A'):
        self.dirt = dirt # {'A': 'Dirty', 'B': 'Clean'}
        self.location = location
        self.score = 0

    def percept(self):
        return [self.location, self.dirt[self.location]]

    def apply(self, action):
        if action == 'Suck':
            self.dirt[self.location] = 'Clean'
        elif action == 'Right':
            self.location = 'B'
        else:
            self.location = 'A'

        # update score
        self.score += sum(1 for sq in self.dirt if self.dirt[sq] == 'Clean')

world = VacuumWorld({'A': 'Dirty', 'B': 'Dirty'})
for step in range(5):
    percept = world.percept()          # read
    action = vacuum_agent(percept)     # decide
    print(f"Step {step}: percept={percept} -> action={action}")
    world.apply(action)                # execute + update
print("Final:", world.dirt, "Score:", world.score)

Step 0: percept=['A', 'Dirty'] -> action=Suck
Step 1: percept=['A', 'Clean'] -> action=Right
Step 2: percept=['B', 'Dirty'] -> action=Suck
Step 3: percept=['B', 'Clean'] -> action=Left
Step 4: percept=['A', 'Clean'] -> action=Right
Final: {'A': 'Clean', 'B': 'Clean'} Score: 8


# Lab 2

## Exercise 1

1. State representation
- We only need to track the number of missionaries, number of cannibals at the near bank and the boat position: `(m, c, b)`.
- The start state is `(3, 3, 1)` and the goal state is `(0, 0, 0)`.

2. Operators
- Five possible actions: move 1 missionary/cannibal, 2 missionaries/cannibals, or 1 each.
- If near bank then subtract from `m` and `c` and flip `b` to 0.

3. Legality test
- A state is only valid if `0 <= m, c <= 3` and on each bank, missionaries are not outnumbered.

4. Goal test
- State is equal to `(0, 0, 0)`.

In [12]:
actions = [(1,0), (0,1), (1,1), (2,0), (0,2)]

def is_valid(state):
    m, c, _ = state
    if not (0 <= m <= 3 and 0 <= c <= 3):
        return False
    if m and m < c:          # near bank
        return False
    om, oc = 3 - m, 3 - c
    if om and om < oc:       # far bank
        return False
    return True
    
def solve(state, path, visited):
    if state == (0, 0, 0):
        return path
    
    visited = visited | {state}
    
    for dm, dc in actions:
        sign = -1 if state[2] == 1 else 1
        new_state = (state[0] + sign*dm, state[1] + sign*dc, 0 if state[2] == 1 else 1)
        if is_valid(new_state) and new_state not in visited:
            result = solve(new_state, path + [new_state], visited)
            if result:
                return result          
    return None                        
        

steps = solve((3, 3, 1), [(3, 3, 1)], set())
print(steps)

[(3, 3, 1), (2, 2, 0), (3, 2, 1), (3, 0, 0), (3, 1, 1), (1, 1, 0), (2, 2, 1), (0, 2, 0), (0, 3, 1), (0, 1, 0), (1, 1, 1), (0, 0, 0)]


With heuristics:

In [14]:
def heuristic(state):
    # people still on the starting bank; fewer left = closer to the goal
    m, c, _ = state
    return m + c

def solve(state, path, visited):
    if state == (0, 0, 0):
        return path
    visited = visited | {state}

    # 1. collect valid, unvisited successors
    successors = []
    for dm, dc in actions:
        sign = -1 if state[2] == 1 else 1
        new_state = (state[0] + sign*dm, state[1] + sign*dc, 0 if state[2] == 1 else 1)
        if is_valid(new_state) and new_state not in visited:
            successors.append(new_state)

    # 2. try the most promising first
    successors.sort(key=heuristic)

    for new_state in successors:
        result = solve(new_state, path + [new_state], visited)
        if result:
            return result
    return None

steps = solve((3, 3, 1), [(3, 3, 1)], set())
print(steps)

[(3, 3, 1), (2, 2, 0), (3, 2, 1), (3, 0, 0), (3, 1, 1), (1, 1, 0), (2, 2, 1), (0, 2, 0), (0, 3, 1), (0, 1, 0), (1, 1, 1), (0, 0, 0)]


## Exercise 2

$x_1 + x_2 + x_3 = 10$

$x_3 + x_6 = 8$

$x_4 + x_5 + x_6 = 11$

$x_1 + x_4 = 9$

In [15]:
def in_domain(n):
    return 1 <= n <= 6

state_no = 0

def solve():
    global state_no
    for x1 in range(1, 7):
        x4 = 9 - x1                              # forced (IV)
        state_no += 1; s = state_no
        if not in_domain(x4) or x4 == x1:
            print(f"S{s}: x1={x1} -> x4={x4}  DEAD END (x4 invalid)")
            continue
        print(f"S{s}: x1={x1} -> x4={x4}  ACTIVE")
        used = {x1, x4}
        for x3 in range(1, 7):
            if x3 in used:
                continue
            state_no += 1; s2 = state_no
            x6 = 8 - x3                          # forced (II)
            x2 = 10 - x1 - x3                    # forced (I)
            x5 = 11 - x4 - x6                    # forced (III)
            vals = [x1, x2, x3, x4, x5, x6]
            if all(in_domain(n) for n in vals) and len(set(vals)) == 6:
                print(f"    S{s2}: x3={x3} -> x2={x2}, x5={x5}, x6={x6}  GOAL {vals}")
                return vals
            print(f"    S{s2}: x3={x3} -> x6={x6}, x2={x2}, x5={x5}  DEAD END, back to S{s}")
        print(f"  x3 exhausted under S{s}: DEAD END, back to root")
    return None

print("Solution:", solve())

S1: x1=1 -> x4=8  DEAD END (x4 invalid)
S2: x1=2 -> x4=7  DEAD END (x4 invalid)
S3: x1=3 -> x4=6  ACTIVE
    S4: x3=1 -> x6=7, x2=6, x5=-2  DEAD END, back to S3
    S5: x3=2 -> x6=6, x2=5, x5=-1  DEAD END, back to S3
    S6: x3=4 -> x6=4, x2=3, x5=1  DEAD END, back to S3
    S7: x3=5 -> x6=3, x2=2, x5=2  DEAD END, back to S3
  x3 exhausted under S3: DEAD END, back to root
S8: x1=4 -> x4=5  ACTIVE
    S9: x3=1 -> x6=7, x2=5, x5=-1  DEAD END, back to S8
    S10: x3=2 -> x6=6, x2=4, x5=0  DEAD END, back to S8
    S11: x3=3 -> x6=5, x2=3, x5=1  DEAD END, back to S8
    S12: x3=6 -> x6=2, x2=0, x5=4  DEAD END, back to S8
  x3 exhausted under S8: DEAD END, back to root
S13: x1=5 -> x4=4  ACTIVE
    S14: x3=1 -> x6=7, x2=4, x5=0  DEAD END, back to S13
    S15: x3=2 -> x2=3, x5=1, x6=6  GOAL [5, 3, 2, 4, 1, 6]
Solution: [5, 3, 2, 4, 1, 6]
